# Rank별 사업자 Anomaly Labeling Tool

M971 합계 순위 기준으로 PLMN을 선택하고, 전체 트렌드 확인 → **그래프에서 줌인** → **anomaly 구간** 라벨링을 수행합니다.

- 라벨: `labels/{PLMN}_labels.json` (이후 수정·삭제 가능)
- 라벨 형식: 항상 **구간** / tag=`anomaly`

**추천 흐름**
1. 클릭모드 **`구간 라벨(2클릭)`**에서 시작점·끝점을 눌러 바로 추가
2. 드래그로 줌하거나 `이동(팬)` / `◀` `▶` / `전체`로 화면 이동
3. `Save Labels`로 파일 저장


In [1]:
import importlib
import os
import sys
import time

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

sys.path.insert(0, os.path.abspath("."))

# tool.py를 고친 뒤 커널을 재시작하지 않아도 되도록 항상 새로 읽는다
import tool as _tool

importlib.reload(_tool)

from tool import (
    LABEL_HIGHLIGHT_NAME,
    PENDING_ANCHOR_NAME,
    VALUE_CURSOR_NAME,
    KeyboardNavigator,
    PlotResizer,
    add_label,
    as_figure_widget,
    build_figure,
    clamp_time_range,
    data_time_bounds,
    display_plmn,
    format_kst,
    label_highlight_overlays,
    label_line,
    load_labels,
    load_or_build_ranking,
    load_plmn,
    metric_columns,
    metric_original,
    minmax_indices,
    parse_time,
    pending_anchor_annotation,
    pending_anchor_shape,
    plmn_original,
    plot_series_window,
    ranked_hover_html,
    remove_label,
    save_labels,
    to_plot_time,
    to_plot_times,
    value_cursor_overlays,
)

# 전체 PLMN (순위 CSV / 전처리 캐시 기준)
rank_df = load_or_build_ranking(top_n=None)
print(f"순위 로드: {len(rank_df)}개 PLMN (전체)")
print(
    f"mapping: PLMN={'on' if plmn_original('P0480') else 'off'}, "
    f"metric={'on' if metric_original('M971') else 'off'}"
)


순위 로드: 1143개 PLMN (전체)
mapping: PLMN=on, metric=on


## Labeling UI

- **줌**: 클릭모드=`줌` → 그래프에서 좌우 드래그. `전체` 또는 더블클릭으로 해제
- **이동**: 클릭모드=`이동(팬)` 드래그, 또는 `◀` / `▶`
- **값 탐색**: 클릭모드=`값 탐색(클릭+←→)` → 시점 클릭 후 키보드 `←` / `→`
- **구간 라벨**: 클릭모드=`구간 라벨(2클릭)` → 시작점·끝점 클릭 (항상 anomaly)
- **값 확인**: 커서 올리면 아래 패널에 값 내림차순 표시


In [2]:
state = {
    "df": None,
    "doc": None,
    "plmn": None,
    "rank": None,
    "metrics": [],
    "zoom_start": None,
    "zoom_end": None,
    "figw": None,
    "fig_metrics": [],
    "label_range_anchor": None,
    "highlight_id": None,
    "value_cursor_pos": None,
    "_syncing_range": False,
    "_syncing_list": False,
    "_last_click": (None, 0.0),
}

MAX_POINTS = 1500

plmn_options = [
    (
        f"#{int(r.rank):03d}  {display_plmn(r.PLMN)}  ({int(r.M971_sum):,})",
        r.PLMN,
    )
    for r in rank_df.itertuples(index=False)
]

w_plmn = widgets.Dropdown(options=plmn_options, description="PLMN", layout=widgets.Layout(width="560px"))
w_load = widgets.Button(description="Load", button_style="primary")
w_prev = widgets.Button(description="◀ Prev")
w_next = widgets.Button(description="Next ▶")

w_click_mode = widgets.ToggleButtons(
    options=[
        ("줌", "zoom"),
        ("이동(팬)", "pan"),
        ("값 탐색(클릭+←→)", "inspect"),
        ("구간 라벨(2클릭)", "label_range"),
    ],
    value="zoom",
    description="",
    layout=widgets.Layout(width="auto"),
)
w_click_mode.style.button_width = "145px"
w_keyboard = KeyboardNavigator()
w_resizer = PlotResizer()

w_reset_zoom = widgets.Button(description="전체", layout=widgets.Layout(width="70px"))
w_pan_left = widgets.Button(
    description="◀",
    tooltip="현재 줌 창을 왼쪽(과거)으로 절반 이동",
    layout=widgets.Layout(width="50px"),
)
w_pan_right = widgets.Button(
    description="▶",
    tooltip="현재 줌 창을 오른쪽(미래)으로 절반 이동",
    layout=widgets.Layout(width="50px"),
)
w_cancel_range = widgets.Button(
    description="시작점 취소",
    button_style="warning",
    layout=widgets.Layout(width="110px", display="none"),
    tooltip="선택한 구간 시작점을 취소합니다",
)

w_note = widgets.Text(description="Note", layout=widgets.Layout(width="420px"))

w_save = widgets.Button(description="Save Labels", button_style="warning")
w_reload = widgets.Button(description="Reload Saved")

w_label_list = widgets.Select(
    options=[],
    rows=6,
    layout=widgets.Layout(width="100%"),
)
w_clear_selection = widgets.Button(
    description="라벨 선택해제",
    tooltip="선택한 라벨과 그래프 강조를 해제합니다",
)
w_delete = widgets.Button(description="선택 라벨 삭제", button_style="danger")
w_zoom_to_selected = widgets.Button(
    description="선택 라벨로 줌",
    button_style="info",
    tooltip="선택한 라벨 구간이 보이도록 그래프를 확대합니다",
)

w_status = widgets.HTML(value="")

fig_box = widgets.Box(
    layout=widgets.Layout(
        display="flex",
        flex_flow="column",
        align_items="stretch",
        width="100%",
        max_width="100%",
        min_width="0px",
        overflow="hidden",
    ),
)
w_hover = widgets.HTML(
    value="<i>그래프에 커서를 올리면 이 시점의 특성값이 <b>내림차순</b>으로 표시됩니다.</i>",
    layout=widgets.Layout(width="100%"),
)
w_hover_box = widgets.Box(
    [w_hover],
    layout=widgets.Layout(
        height="240px",
        width="100%",
        overflow="auto",
        border="1px solid #ddd",
        padding="8px",
    ),
)


def _selected_metrics():
    return state["metrics"] if state["df"] is not None else []


def _selected_label_id():
    return w_label_list.value


def _label_by_id(label_id):
    for item in (state["doc"] or {}).get("labels", []):
        if item["id"] == label_id:
            return item
    return None


def _refresh_label_list():
    items = (state["doc"] or {}).get("labels", [])
    keep = w_label_list.value
    state["_syncing_list"] = True
    w_label_list.options = [(label_line(x), x["id"]) for x in items]
    ids = [x["id"] for x in items]
    w_label_list.value = keep if keep in ids else (ids[0] if ids else None)
    state["_syncing_list"] = False


def _fmt_ts(ts) -> str:
    return format_kst(ts)


def _parse_click_x(x_val):
    return parse_time(x_val)


def _set_zoom_range(start_ts, end_ts, apply_to_fig=True):
    if state["df"] is None:
        return
    tmin, tmax = data_time_bounds(state["df"])
    start_ts, end_ts = clamp_time_range(start_ts, end_ts, tmin, tmax)
    state["zoom_start"] = start_ts
    state["zoom_end"] = end_ts
    if apply_to_fig and state["figw"] is not None:
        with state["figw"].batch_update():
            state["figw"].layout.xaxis.autorange = False
            state["figw"].layout.xaxis.range = [
                to_plot_time(start_ts),
                to_plot_time(end_ts),
            ]


def _current_x_range():
    if state["figw"] is not None and state["figw"].layout.xaxis.range:
        try:
            return (
                parse_time(state["figw"].layout.xaxis.range[0]),
                parse_time(state["figw"].layout.xaxis.range[1]),
            )
        except Exception:
            pass
    if state["zoom_start"] is not None and state["zoom_end"] is not None:
        return state["zoom_start"], state["zoom_end"]
    if state["df"] is None:
        return None, None
    return state["df"]["time"].min(), state["df"]["time"].max()


def _shift_zoom(direction: int, fraction: float = 0.5):
    if state["df"] is None:
        return
    start_ts, end_ts = _current_x_range()
    if start_ts is None or end_ts is None:
        return
    width = end_ts - start_ts
    if width <= pd.Timedelta(0):
        return
    delta = width * fraction * direction
    _set_zoom_range(start_ts + delta, end_ts + delta, apply_to_fig=True)
    _resample_to_view()


def _update_status(extra=""):
    w_status.value = extra or ""


def _sync_cancel_button():
    show = (
        w_click_mode.value == "label_range"
        and state["label_range_anchor"] is not None
    )
    w_cancel_range.layout.display = "inline-flex" if show else "none"


def _on_xaxis_range(layout, xrange):
    if state["_syncing_range"] or xrange is None or state["df"] is None:
        return
    try:
        start_ts = parse_time(xrange[0])
        end_ts = parse_time(xrange[1])
    except Exception:
        return
    tmin, tmax = data_time_bounds(state["df"])
    clamped_start, clamped_end = clamp_time_range(start_ts, end_ts, tmin, tmax)
    if clamped_start != start_ts or clamped_end != end_ts:
        state["_syncing_range"] = True
        if state["figw"] is not None:
            with state["figw"].batch_update():
                state["figw"].layout.xaxis.autorange = False
                state["figw"].layout.xaxis.range = [
                    to_plot_time(clamped_start),
                    to_plot_time(clamped_end),
                ]
        state["_syncing_range"] = False
        start_ts, end_ts = clamped_start, clamped_end
    state["zoom_start"] = start_ts
    state["zoom_end"] = end_ts
    _resample_to_view()


def _resample_to_view():
    figw = state["figw"]
    df = state["df"]
    cols = state.get("fig_metrics") or []
    if figw is None or df is None or not cols:
        return
    start_ts, end_ts = _current_x_range()
    series = plot_series_window(df, cols, start_ts, end_ts, MAX_POINTS)
    with figw.batch_update():
        for i, col in enumerate(cols):
            if i >= len(figw.data):
                break
            x, y = series[col]
            figw.data[i].x = x
            figw.data[i].y = y
        anchor_i = len(cols)
        if anchor_i < len(figw.data) and figw.data[anchor_i].name == "values":
            view = df[(df["time"] >= start_ts) & (df["time"] <= end_ts)]
            if len(view):
                top = view[cols].max(axis=1)
                idx = minmax_indices(top.to_numpy(), MAX_POINTS)
                figw.data[anchor_i].x = to_plot_times(view["time"].to_numpy()[idx])
                figw.data[anchor_i].y = top.to_numpy()[idx]


def _clear_pending_anchor():
    figw = state["figw"]
    if figw is None:
        return
    shapes = tuple(s for s in figw.layout.shapes if s.name != PENDING_ANCHOR_NAME)
    notes = tuple(a for a in figw.layout.annotations if a.name != PENDING_ANCHOR_NAME)
    with figw.batch_update():
        figw.layout.shapes = shapes
        figw.layout.annotations = notes


def _show_pending_anchor(ts):
    figw = state["figw"]
    if figw is None:
        return
    _clear_pending_anchor()
    with figw.batch_update():
        figw.layout.shapes = tuple(figw.layout.shapes) + (pending_anchor_shape(ts),)
        figw.layout.annotations = tuple(figw.layout.annotations) + (
            pending_anchor_annotation(ts),
        )


def _clear_label_highlight():
    figw = state["figw"]
    if figw is None:
        return
    shapes = tuple(s for s in figw.layout.shapes if s.name != LABEL_HIGHLIGHT_NAME)
    notes = tuple(a for a in figw.layout.annotations if a.name != LABEL_HIGHLIGHT_NAME)
    with figw.batch_update():
        figw.layout.shapes = shapes
        figw.layout.annotations = notes


def _highlight_label(label_id):
    state["highlight_id"] = label_id
    figw = state["figw"]
    if figw is None:
        return
    _clear_label_highlight()
    item = _label_by_id(label_id)
    if item is None:
        return
    shapes, notes = label_highlight_overlays(item)
    with figw.batch_update():
        figw.layout.shapes = tuple(figw.layout.shapes) + tuple(shapes)
        figw.layout.annotations = tuple(figw.layout.annotations) + tuple(notes)


def _clear_value_cursor():
    figw = state["figw"]
    if figw is None:
        return
    shapes = tuple(s for s in figw.layout.shapes if s.name != VALUE_CURSOR_NAME)
    notes = tuple(a for a in figw.layout.annotations if a.name != VALUE_CURSOR_NAME)
    with figw.batch_update():
        figw.layout.shapes = shapes
        figw.layout.annotations = notes


def _select_value_pos(pos):
    df = state["df"]
    if df is None or not len(df):
        return
    pos = max(0, min(int(pos), len(df) - 1))
    state["value_cursor_pos"] = pos
    row = df.iloc[pos]
    _clear_value_cursor()
    if state["figw"] is not None:
        shape, note = value_cursor_overlays(row["time"])
        with state["figw"].batch_update():
            state["figw"].layout.shapes = tuple(state["figw"].layout.shapes) + (shape,)
            state["figw"].layout.annotations = tuple(state["figw"].layout.annotations) + (note,)
    _update_hover_panel(row["time"])
    _update_status(f"값 탐색: {format_kst(row['time'])} · ←/→ 키로 이동")


def on_keyboard_navigation(change):
    if w_click_mode.value != "inspect" or state["value_cursor_pos"] is None:
        return
    step = -1 if w_keyboard.direction == "left" else 1
    _select_value_pos(state["value_cursor_pos"] + step)


def _commit_label(start_ts, end_ts):
    """Always add an anomaly range label."""
    if state["doc"] is None:
        return None
    start_ts = parse_time(start_ts)
    end_ts = parse_time(end_ts)
    if end_ts < start_ts:
        start_ts, end_ts = end_ts, start_ts
    before = {x["id"] for x in state["doc"].get("labels", [])}
    add_label(
        state["doc"],
        kind="range",
        tag="anomaly",
        start=start_ts,
        end=end_ts,
        metrics=["ALL"],
        note=w_note.value.strip(),
    )
    after = [x for x in state["doc"]["labels"] if x["id"] not in before]
    lid = after[0]["id"] if after else "?"
    state["highlight_id"] = lid
    render(rebuild_figure=True)
    w_label_list.value = lid if lid in [x["id"] for x in state["doc"]["labels"]] else None
    when = f"{format_kst(start_ts)} → {format_kst(end_ts)}"
    _update_status(
        f"<span style='color:green'>✔ [구간] anomaly 추가됨 "
        f"({lid}) {when} — Save Labels로 저장</span>"
    )
    _sync_cancel_button()
    return lid


def _on_trace_click(trace, points, selector):
    if not points.point_inds:
        return
    ts = _parse_click_x(points.xs[0])
    mode = w_click_mode.value
    last_ts, last_at = state.get("_last_click", (None, 0.0))
    now = time.monotonic()
    if last_ts == ts and now - last_at < 0.6:
        return
    state["_last_click"] = (ts, now)

    if mode == "inspect":
        pos = int((state["df"]["time"] - ts).abs().to_numpy().argmin())
        _select_value_pos(pos)
        return
    if mode != "label_range":
        return

    anchor = state["label_range_anchor"]
    if anchor is None:
        state["label_range_anchor"] = ts
        _show_pending_anchor(ts)
        _update_status(f"① 구간 시작: {format_kst(ts)} → ② 끝점을 클릭하세요")
        _sync_cancel_button()
    else:
        state["label_range_anchor"] = None
        _clear_pending_anchor()
        _commit_label(anchor, ts)
        _sync_cancel_button()


def _update_hover_panel(ts):
    if state["df"] is None:
        return
    nearest = (state["df"]["time"] - ts).abs().idxmin()
    row = state["df"].loc[nearest]
    w_hover.value = ranked_hover_html(
        row["time"], row, _selected_metrics() or state["metrics"]
    )


def _on_trace_hover(trace, points, selector):
    if not points.point_inds:
        return
    _update_hover_panel(_parse_click_x(points.xs[0]))


def _attach_figure_callbacks(figw):
    figw.layout.on_change(_on_xaxis_range, "xaxis.range")
    click_traces = [tr for tr in figw.data if getattr(tr, "name", None) == "values"]
    if not click_traces:
        click_traces = list(figw.data[:1])
    for tr in click_traces:
        try:
            tr.on_click(_on_trace_click)
        except Exception:
            pass
    for tr in figw.data:
        try:
            tr.on_hover(_on_trace_hover)
        except Exception:
            pass


def render(rebuild_figure=True):
    if state["df"] is None or state["doc"] is None:
        return

    metrics = _selected_metrics()
    title = f"#{state['rank']:03d} {display_plmn(state['plmn'])} | labels={len(state['doc']['labels'])}"

    if rebuild_figure:
        fig = build_figure(
            state["df"],
            state["doc"],
            metrics=metrics,
            start=state["zoom_start"],
            end=state["zoom_end"],
            title=title,
            hover_values=False,
            max_points=MAX_POINTS,
        )
        fig.update_layout(
            dragmode="pan" if w_click_mode.value in ("pan", "inspect") else "zoom"
        )
        fig.update_yaxes(fixedrange=True)
        figw = as_figure_widget(fig)
        _attach_figure_callbacks(figw)
        state["figw"] = figw
        state["fig_metrics"] = list(metrics)
        fig_box.children = (figw,)
        w_resizer.token += 1
        if state["label_range_anchor"] is not None:
            _show_pending_anchor(state["label_range_anchor"])
        if state.get("highlight_id"):
            _highlight_label(state["highlight_id"])
        if state.get("value_cursor_pos") is not None:
            _select_value_pos(state["value_cursor_pos"])

    _refresh_label_list()
    _sync_cancel_button()


def on_load(_=None):
    plmn = w_plmn.value
    rank = int(rank_df.loc[rank_df["PLMN"] == plmn, "rank"].iloc[0])
    _update_status(f"<i>Loading {plmn}...</i>")
    df = load_plmn(plmn)
    metrics = metric_columns(df)
    doc = load_labels(plmn, rank=rank)

    state.update(
        df=df,
        doc=doc,
        plmn=plmn,
        rank=rank,
        metrics=metrics,
        zoom_start=None,
        zoom_end=None,
        label_range_anchor=None,
        highlight_id=None,
        value_cursor_pos=None,
        figw=None,
    )
    tmin, tmax = data_time_bounds(df)
    state["zoom_start"] = tmin
    state["zoom_end"] = tmax
    render(rebuild_figure=True)
    _update_status("")


def on_prev(_):
    opts = [v for _, v in plmn_options]
    i = opts.index(w_plmn.value)
    if i > 0:
        w_plmn.value = opts[i - 1]
        on_load()


def on_next(_):
    opts = [v for _, v in plmn_options]
    i = opts.index(w_plmn.value)
    if i < len(opts) - 1:
        w_plmn.value = opts[i + 1]
        on_load()


def on_reset_zoom(_):
    if state["df"] is None:
        return
    state["label_range_anchor"] = None
    _clear_pending_anchor()
    tmin, tmax = data_time_bounds(state["df"])
    state["zoom_start"] = tmin
    state["zoom_end"] = tmax
    if state["figw"] is not None:
        with state["figw"].batch_update():
            state["figw"].layout.xaxis.autorange = False
            state["figw"].layout.xaxis.range = [
                to_plot_time(tmin),
                to_plot_time(tmax),
            ]
            state["figw"].layout.yaxis.fixedrange = True
    _resample_to_view()
    _sync_cancel_button()


def on_save(_):
    if state["doc"] is None:
        return
    path = save_labels(state["doc"])
    render(rebuild_figure=False)
    _update_status(f"<span style='color:green'>Saved: {path}</span>")


def on_reload(_):
    if state["plmn"] is None:
        return
    state["doc"] = load_labels(state["plmn"], rank=state["rank"])
    render(rebuild_figure=True)


def on_delete(_):
    label_id = _selected_label_id()
    if state["doc"] is None or not label_id:
        return
    remove_label(state["doc"], label_id)
    if state.get("highlight_id") == label_id:
        state["highlight_id"] = None
    render(rebuild_figure=True)
    _update_status(f"삭제됨: {label_id} — Save Labels로 저장")


def on_label_select(change):
    if state.get("_syncing_list"):
        return
    label_id = change["new"]
    if not label_id:
        state["highlight_id"] = None
        _clear_label_highlight()
        return
    _highlight_label(label_id)


def on_clear_selection(_):
    state["highlight_id"] = None
    w_label_list.value = None
    _clear_label_highlight()
    _update_status("라벨 선택이 해제되었습니다.")


def on_zoom_to_selected(_):
    item = _label_by_id(_selected_label_id())
    if item is None or state["df"] is None:
        return
    start_ts = parse_time(item["start"])
    end_ts = parse_time(item["end"])
    span = end_ts - start_ts
    pad = span / 2 if span > pd.Timedelta(0) else pd.Timedelta(hours=6)
    _set_zoom_range(start_ts - pad, end_ts + pad, apply_to_fig=True)
    _resample_to_view()
    _highlight_label(item["id"])


def on_cancel_range(_):
    state["label_range_anchor"] = None
    _clear_pending_anchor()
    _update_status("구간 시작점 취소됨")
    _sync_cancel_button()


def on_click_mode_change(change):
    state["label_range_anchor"] = None
    _clear_pending_anchor()
    w_keyboard.enabled = change["new"] == "inspect"
    if change["new"] != "inspect":
        state["value_cursor_pos"] = None
        _clear_value_cursor()
    if state["figw"] is not None:
        if change["new"] in ("pan", "inspect"):
            state["figw"].layout.dragmode = "pan"
        else:
            state["figw"].layout.dragmode = "zoom"
        state["figw"].layout.yaxis.fixedrange = True
    _sync_cancel_button()
    if change["new"] == "zoom":
        _update_status("그래프에서 좌우로 드래그해 시간축을 확대하세요.")
    elif change["new"] == "pan":
        _update_status("좌우로 드래그해 시간축을 이동하세요.")
    elif change["new"] == "inspect":
        _update_status("그래프에서 시점을 클릭한 뒤 ←/→ 키로 값을 탐색하세요.")
    else:
        _update_status("① 시작점 클릭 → ② 끝점 클릭으로 구간 라벨 추가")


def on_pan_left(_):
    _shift_zoom(-1)


def on_pan_right(_):
    _shift_zoom(1)


w_load.on_click(on_load)
w_prev.on_click(on_prev)
w_next.on_click(on_next)
w_reset_zoom.on_click(on_reset_zoom)
w_pan_left.on_click(on_pan_left)
w_pan_right.on_click(on_pan_right)
w_save.on_click(on_save)
w_reload.on_click(on_reload)
w_delete.on_click(on_delete)
w_clear_selection.on_click(on_clear_selection)
w_zoom_to_selected.on_click(on_zoom_to_selected)
w_label_list.observe(on_label_select, names="value")
w_click_mode.observe(on_click_mode_change, names="value")
w_keyboard.observe(on_keyboard_navigation, names="sequence")
w_cancel_range.on_click(on_cancel_range)
click_controls = widgets.HBox(
    [
        w_click_mode,
        w_pan_left,
        w_pan_right,
        w_reset_zoom,
        w_cancel_range,
        w_keyboard,
        w_resizer,
    ],
    layout=widgets.Layout(
        width="100%",
        max_width="100%",
        flex_flow="row nowrap",
        align_items="center",
        border="1px solid #e0e0e0",
        padding="6px",
        margin="0 0 4px 0",
        overflow="hidden",
    ),
)

hover_panel = widgets.VBox(
    [widgets.HTML("<b>이 시점 특성값 (값 내림차순)</b>"), w_hover_box],
    layout=widgets.Layout(width="100%"),
)

label_panel = widgets.VBox(
    [
        widgets.HTML(
            "<span style='font-size:12px;color:#666'>"
            "라벨은 항상 <b>anomaly 구간</b>입니다. 그래프에서 시작·끝을 클릭해 추가하세요.</span>"
        ),
        widgets.HBox([w_note, w_save, w_reload]),
        widgets.HTML(
            "<b style='font-size:13px'>저장된 라벨 (한 줄 = 라벨 1개)</b>"
            "<span style='font-size:12px;color:#666'> — 한 줄을 클릭하면 그래프에서 "
            "노란색으로 강조됩니다.</span>"
        ),
        w_label_list,
        widgets.HBox([w_clear_selection, w_zoom_to_selected, w_delete]),
    ],
    layout=widgets.Layout(width="100%"),
)

ui = widgets.VBox(
    layout=widgets.Layout(width="100%", max_width="100%"),
    children=[
        widgets.HTML("<h3 style='margin:4px 0'>1) 사업자 선택</h3>"),
        widgets.HBox([w_prev, w_plmn, w_next, w_load]),
        widgets.HTML("<h3 style='margin:10px 0 4px'>2) 그래프</h3>"),
        click_controls,
        fig_box,
        w_status,
        hover_panel,
        widgets.HTML("<h3 style='margin:10px 0 4px'>3) 라벨 추가 / 수정</h3>"),
        label_panel,
    ],
)

display(ui)


## 사용 팁

| 목적 | 방법 |
|------|------|
| 이상 구간 확대 | 그래프에서 **드래그 박스 줌** (상하 무시, 좌우만) |
| **좌우 이동** | 클릭모드=`이동(팬)` 후 드래그, 또는 `◀` / `▶` |
| **구간 라벨** | 클릭모드=`구간 라벨` → 시작·끝 2클릭 (항상 anomaly) |
| 줌 해제 | 그래프 더블클릭 또는 `전체` |
| **라벨 확인** | 목록에서 한 줄 클릭 → 강조 / `선택 라벨로 줌` / 삭제 |
| 저장 | `Save Labels` |


## (선택) 코드로 일괄 수정

In [3]:
# doc = load_labels("P0480", rank=1)
# print("\n".join(label_line(x) for x in doc["labels"]))
# add_label(doc, kind="range", tag="anomaly",
#           start="2026-05-10 00:00:00+00:00",
#           end="2026-05-10 12:00:00+00:00",
#           note="수동 구간")
# save_labels(doc)